In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE DEV_EV_ANALYTICS._98_LOGGING.STATIONS_LOAD_ERRORS (

    JOB_ID STRING,
    ERROR_MESSAGE STRING,
    ERROR_TYPE STRING,
    ERROR_TIME TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()

);

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, current_timestamp, lit
from datetime import datetime


session = get_active_session()

# Set the database and schema context
session.sql("USE DATABASE DEV_EV_ANALYTICS").collect()
session.sql("USE SCHEMA \"_00_STAGING\"").collect()
session.sql("USE WAREHOUSE XS_WAREHOUSE").collect()

# Generate JOB_ID
job_id = "JOB_ID" + datetime.now().strftime("%Y%m%d%H%M%S")

try:

    df_json = session.read \
        .option("format_name", "json_format") \
        .json('@"DEV_EV_ANALYTICS"."_00_STAGING"."EV_JSON_STAGE"/EV_Roam_charging_stations_data.json')
    
    
    df_stations = df_json.select(
        col("$1")["OBJECTID"].cast("string").alias("OBJECTID"),
        col("$1")["NAME"].cast("string").alias("NAME"),
        col("$1")["OPERATOR"].cast("string").alias("OPERATOR"),
        col("$1")["OWNER"].cast("string").alias("OWNER"),
        col("$1")["ADDRESS"].cast("string").alias("ADDRESS"),
        col("$1")["is24Hours"].cast("string").alias("IS_24_HOURS"),
        col("$1")["carParkCount"].cast("integer").alias("CAR_PARK_COUNT"),
        col("$1")["hasCarparkCost"].cast("string").alias("HAS_CARPARK_COST"),
        col("$1")["maxTimeLimit"].cast("string").alias("MAX_TIME_LIMIT"),
        col("$1")["hasTouristAttraction"].cast("string").alias("HAS_TOURIST_ATTRACTION"),
        col("$1")["latitude"].cast("float").alias("LATITUDE"),
        col("$1")["longitude"].cast("float").alias("LONGITUDE"),
        col("$1")["currentType"].cast("string").alias("CURRENT_TYPE"),
        col("$1")["dateFirstOperational"].cast("string").alias("DATE_FIRST_OPERATIONAL"),
        col("$1")["numberOfConnectors"].cast("integer").alias("NUMBER_OF_CONNECTORS"),
        col("$1")["connectorsList"].cast("string").alias("CONNECTORS_LIST"),
        col("$1")["hasChargingCost"].cast("string").alias("HAS_CHARGING_COST"),
        col("$1")["GlobalID"].cast("string").alias("GLOBAL_ID"),
        # Adding Metadata here
        current_timestamp().alias("_LOADED_AT"),
        lit(job_id).alias("JOB_ID")
    )
    
    df_stations.write \
        .mode("overwrite") \
        .save_as_table('"DEV_EV_ANALYTICS"."_01_BRONZE"."STATIONS_TBL"')

    print("Load Successful")

except Exception as e:

    error_message = str(e)
    error_type = type(e).__name__

    print("ERROR:", error_message)


    # Insert into logging table
    session.sql(f"""
        INSERT INTO DEV_EV_ANALYTICS._98_LOGGING.STATIONS_LOAD_ERRORS
        (JOB_ID, ERROR_MESSAGE, ERROR_TYPE)
        VALUES (
            '{job_id}',
            $$ {error_message} $$,
            '{error_type}'
        )
    """).collect()

    raise